# Stage 3: evaluate the adapted model (Qwen3-1.7B + 10,000 Urdu tokens)

Scores three models on **the same texts** with the Stage 2 protocol: the base model, the extended model before training, and the adapted model from notebook 07. Bits per character on held-out Urdu, FLORES Urdu and FLORES English, and Belebele accuracy (Urdu and English, 900 questions each, with 95% intervals). It then prints the Stage 3 success checks as PASS or FAIL.

Nothing is trained here. It takes about an hour on a T4.

Settings before running: Accelerator **GPU T4**, Internet **on** (downloads the base model, Belebele and FLORES), Input **Add Input > Notebooks**: the output of `03_extend_export_qwen3` **and** the output of `07_adapt_main_qwen3`. No secret is needed. Use *Save Version > Save & Run All (Commit)*.

Models run in float32 (as in Stage 2). The adapted model is stored as bfloat16; loading it as float32 is exact, so the comparison is fair.

In [ ]:
# Install zubaan at one exact commit. Do not edit the next line except to change the commit.
COMMIT = "8d16b47"
REPO = f"git+https://github.com/Hashir-Aziz123/zubaan.git@{COMMIT}"
!pip install -q "zubaan[audit,train] @ {REPO}"
# Kaggle may already hold an older copy with the same version number, which pip keeps.
# Force our commit (dependencies are already in place).
!pip install -q --force-reinstall --no-deps "zubaan @ {REPO}"
# Kaggle ships torchao 0.10, which peft refuses ("only versions above 0.16 are supported") when it
# builds LoRA layers. We never use torchao, so remove it; peft then skips its torchao support.
!pip uninstall -y -q torchao
!zubaan version

In [ ]:
# CHECK 1: the installed code is the commit above, not a cached older copy.
# If this fails, use Run > Restart session and run the cells again from the top.
import json
from importlib import metadata

direct = json.loads(metadata.distribution("zubaan").read_text("direct_url.json"))
installed = direct["vcs_info"]["commit_id"]
print("installed commit:", installed)
assert installed.startswith(COMMIT), f"installed {installed}, wanted {COMMIT}"

import zubaan.adapt.pipeline  # noqa: F401

print("OK: code version verified")

import zubaan.adapt.evaluate  # noqa: F401  (the evaluation code is in this commit)
print("OK: evaluation code present")

In [ ]:
import os
import subprocess

# Quiet progress bars: they can overwhelm the notebook front end.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# CHECK 2: a GPU with about 16 GB. Training uses one GPU (fp16: a T4 and a P100 have no BF16).
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
except FileNotFoundError:
    raise SystemExit("No GPU in this session (nvidia-smi is missing). Settings > Accelerator > GPU T4 x2, then run again.") from None
assert r.returncode == 0, "no GPU found: turn on a GPU accelerator in the notebook settings"
print(r.stdout)
gpu_name, cap, mem = [x.strip() for x in r.stdout.splitlines()[0].split(",")]
assert float(mem.split()[0]) >= 14000, f"{gpu_name} has {mem}; need about 16 GB"
if float(cap) < 7.0:
    print("WARNING: compute capability", cap, "has no fp16 tensor cores; this will be slow. A T4 is better.")

import peft
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "torch cannot see the GPU"
print("OK: GPU", gpu_name, "compute capability", cap, mem)

## 1. Find the two local models

In [ ]:
import json

from pathlib import Path

# Two folders written by zubaan come from Kaggle inputs: the extended model before training
# (output of notebook 03) and the adapted model (output of notebook 07). The marker file tells them apart.
INPUT_ROOT = Path("/kaggle/input")
markers = sorted(INPUT_ROOT.rglob("zubaan_extension.json"))
assert markers, "no zubaan model found under /kaggle/input: add the outputs of notebooks 03 and 07 (Add Input > Notebooks)"
trained, untrained = [], []
for m in markers:
    (trained if json.loads(m.read_text()).get("trained") else untrained).append(m.parent)
for name, folders in [("adapted (trained)", trained), ("extended (untrained)", untrained)]:
    print(name, [str(f) for f in folders])
assert len(trained) == 1, f"need exactly one trained model folder, found {len(trained)}: add only notebook 07's output (or remove the others)"
assert len(untrained) >= 1, "need the untrained extended model: add the output of notebook 03"
ADAPTED_DIR, EXTENDED_DIR = trained[0], untrained[0]
for d in (ADAPTED_DIR, EXTENDED_DIR):
    assert (d / "tokenizer.json").exists(), f"no tokenizer.json in {d}"
    assert (d / "model.safetensors").exists() or list(d.glob("model-*.safetensors")), f"no weights in {d}"
print(json.dumps(json.loads((ADAPTED_DIR / "zubaan_extension.json").read_text()), indent=1))

# Both tokenizers must be identical: training never touches the tokenizer.
a = json.loads((ADAPTED_DIR / "tokenizer.json").read_text())
b = json.loads((EXTENDED_DIR / "tokenizer.json").read_text())
assert a == b, "adapted and extended tokenizer.json differ"
print("OK: adapted and extended tokenizers are identical")

## 2. Configuration

In [ ]:
from pathlib import Path

import yaml

BASE = "Qwen/Qwen3-1.7B"
evaluate = {
    "models": {"base": BASE, "extended-untrained": str(EXTENDED_DIR), "adapted": str(ADAPTED_DIR)},
    "untrained": "extended-untrained",
    "adapted": "adapted",
    "heldout": "data/urdu/urdu_heldout.txt",
    "eval_urdu": "data/flores_urd.txt",
    "eval_english": "data/flores_eng.txt",
    # The Stage 2 protocol: 1000 texts of at most 400 characters per set, all 900 Belebele questions, float32.
    "quality_texts": 1000,
    "max_chars": 400,
    "belebele_questions": None,
    "dtype": "float32",
    "device": "cuda",
    "batch_size": 4,
    "max_batch_tokens": 2048,
    "output": "/kaggle/working/adapt_quality.json",
}
config = {
    "urdu_corpus": {"raw_dir": "data/raw", "out_dir": "data/urdu", "flores": "data/flores_urd.txt", "max_mb": 100, "heldout_percent": 1.0},
    "evaluate": evaluate,
}
Path("evaluate.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
print(Path("evaluate.yaml").read_text())

## 3. Data

In [ ]:
# The same evaluation text as Stage 2: FLORES devtest, and the held-out Urdu Wikipedia lines.
!zubaan data flores --out data
!zubaan data urdu --config evaluate.yaml

In [ ]:
import hashlib

meta = json.loads(Path("data/urdu/urdu_meta.json").read_text())
sha = meta["files"]["heldout"]["sha256"]
print("held-out sha256:", sha)
# Stage 2 measured the base model on a file with this hash. A different file would make the rows
# incomparable with the Stage 2 table, so stop.
assert sha == "19c4f1b63b41b8a91cf8aa345d0f0e83945f33e403cc78c1f6f52ed7e7c34ada", "the held-out Urdu file differs from the one Stage 2 used"
print("OK: same held-out file as Stage 2")

## 4. Rehearsal

In [ ]:
# CHECK 3, rehearsal (a few minutes): the adapted model only, on 12 texts and 4 Belebele questions, written
# to a separate file. The full run starts only if this works.
import copy
import math

rh = copy.deepcopy(config)
rh["evaluate"].update({"models": {"adapted": str(ADAPTED_DIR)}, "untrained": None, "adapted": None,
                       "quality_texts": 12, "belebele_questions": 4, "output": "/kaggle/working/rehearsal.json"})
Path("evaluate_rehearsal.yaml").write_text(yaml.safe_dump(rh, sort_keys=False))
Path("/kaggle/working/rehearsal.json").unlink(missing_ok=True)
!zubaan adapt eval --config evaluate_rehearsal.yaml

row = json.loads(Path("/kaggle/working/rehearsal.json").read_text())["variants"]["adapted"]
for name, r in row["bpc"].items():
    assert math.isfinite(r["bits_per_character"]), name
print({k: round(v["bits_per_character"], 2) for k, v in row["bpc"].items()}, row["memory"])
assert row["memory"].get("peak_gpu_gb", 0) < 14, "peak GPU memory too close to 16 GB: lower max_batch_tokens"
print("REHEARSAL PASSED")

## 5. Full evaluation

In [ ]:
# Base (downloaded), extended-untrained and adapted, all on the same texts. Finished models are skipped
# if the session has to be run again.
!zubaan adapt eval --config evaluate.yaml

In [ ]:
# The base model must reproduce its Stage 2 numbers (same texts, same dtype); if it does not, the
# protocol drifted and the comparison below cannot be trusted.
res = json.loads(Path("/kaggle/working/adapt_quality.json").read_text())
base = res["variants"]["base"]
stage2 = {"urdu_heldout": 2.310, "urdu_flores": 2.339, "english_flores": 1.208}
for name, want in stage2.items():
    got = base["bpc"][name]["bits_per_character"]
    print(f"base {name}: now {got:.3f}, Stage 2 {want:.3f}")
    assert abs(got - want) < 0.01, f"base {name} differs from Stage 2 by {got - want:+.3f}"
print("OK: the base model reproduces the Stage 2 table")

Download `adapt_quality.json` from the Output tab into your local `reports/` folder and send it back. Lower bits per character is better; Belebele chance is 0.25. The checks above (PASS or FAIL) are printed at the end of the full run.